# Treino LoRA com Unsloth — Qwen2.5-Coder-1.5B-Instruct (Etapa 4)

> **NÃO rodar ainda** antes de fechar a Etapa 3 (`base-ctx-fix`) e o plano de dados *on-policy*.

- `FastLanguageModel.from_pretrained` (`load_in_4bit=False`, `bf16` em L4/A100, `fp16` em T4)
- Loss **só na resposta** (`train_on_responses_only` com chat template nativo Qwen2.5)
- Teste curto de **200 passos** (`lr=1e-4`) antes dos 3 treinos completos (`5e-5`, `1e-4`, `2e-4`), mesmo seed, checkpoints no Drive com retomada automática.


In [ ]:
# Célula 1: Verificar GPU, nproc e dtype
!nvidia-smi
import os, torch
assert torch.cuda.is_available(), "GPU não disponível!"
gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
DTYPE = torch.bfloat16 if cc[0] >= 8 else torch.float16
USE_BF16 = cc[0] >= 8
print(f"GPU: {gpu_name} | CC: {cc} | dtype={DTYPE} | nproc={os.cpu_count()}")


In [ ]:
# Célula 2: Montar Drive e pastas
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/gograph"
for d in ["data", "out", "bench"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
print("Drive pronto:", DRIVE_ROOT)


In [ ]:
# Célula 3: Instalar Unsloth + dependências e registrar versões
!pip install -q unsloth
import json, torch, unsloth, transformers, trl, peft
VERSIONS = {
    "unsloth": unsloth.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "peft": peft.__version__,
}
print("Versões:", json.dumps(VERSIONS, indent=2))
with open(f"{DRIVE_ROOT}/out/train_versions.json", "w") as f:
    json.dump(VERSIONS, f, indent=2)


In [ ]:
# Célula 4: Deduplicação vs benchmark + cálculo de max_seq_length sem truncamento + split 5%
import json, hashlib, random, math
from transformers import AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
DATA_PATH = f"{DRIVE_ROOT}/data/train_limpo.jsonl"
BENCH_TEST = f"{DRIVE_ROOT}/data/tarefas-teste.jsonl"
BENCH_VAL = f"{DRIVE_ROOT}/data/tarefas-validacao.jsonl"
SEED = 42

tok = AutoTokenizer.from_pretrained(BASE_MODEL)

bench_names = set()
bench_hashes = set()
for bp in [BENCH_TEST, BENCH_VAL]:
    try:
        with open(bp) as f:
            for line in f:
                r = json.loads(line)
                if r.get("name"):
                    bench_names.add((r.get("source", ""), r["name"]))
                if r.get("hash"):
                    bench_hashes.add(r["hash"])
                if r.get("original"):
                    bench_hashes.add(hashlib.sha256(r["original"].strip().encode()).hexdigest())
    except FileNotFoundError:
        pass

raw = []
dropped = 0
with open(DATA_PATH) as f:
    for line in f:
        r = json.loads(line)
        body_hash = hashlib.sha256(r.get("output", "").strip().encode()).hexdigest()
        if r.get("hash") in bench_hashes or body_hash in bench_hashes or (r.get("source", ""), r.get("name", "")) in bench_names:
            dropped += 1
            continue
        raw.append(r)

def fmt_text(ex):
    return tok.apply_chat_template([
        {"role": "system", "content": ex["system"]},
        {"role": "user", "content": ex["input"]},
        {"role": "assistant", "content": ex["output"]},
    ], tokenize=False)

texts = [fmt_text(ex) for ex in raw]
lens = [len(x) for x in tok(texts, truncation=False, padding=False)["input_ids"]]
max_toks = max(lens)
MAX_SEQ_LEN = int(math.ceil(max_toks / 1024.0) * 1024)
assert max_toks <= MAX_SEQ_LEN
print(f"Total: {len(raw)} (removidos vs bench: {dropped}) | Maior exemplo: {max_toks} tokens | MAX_SEQ_LEN={MAX_SEQ_LEN} (0 truncados)")

for ex, t in zip(raw, texts):
    ex["text"] = t

random.Random(SEED).shuffle(raw)
val_n = max(1, int(0.05 * len(raw)))
val_data = raw[:val_n]
train_data = raw[val_n:]
print(f"Treino: {len(train_data)} | Val: {len(val_data)}")


In [ ]:
# Célula 5: Helper de treino Unsloth (loss só na resposta, EscritaCallback, early stopping, retomada)
import time, csv, glob, os
import torch
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import EarlyStoppingCallback, TrainerCallback
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

train_ds = Dataset.from_list([{"text": x["text"]} for x in train_data])
val_ds = Dataset.from_list([{"text": x["text"]} for x in val_data])

class EscritaCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            print(f"[step {state.global_step}] val_loss={metrics.get("eval_loss", 0):.4f}")

def run_training(lr, out_dir, max_steps=-1):
    os.makedirs(out_dir, exist_ok=True)
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LEN,
        dtype=DTYPE,
        load_in_4bit=False,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
    )

    cfg = SFTConfig(
        output_dir=out_dir,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=False,
        num_train_epochs=1,
        max_steps=max_steps,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        seed=SEED,
        bf16=USE_BF16,
        fp16=not USE_BF16,
        logging_steps=25 if max_steps > 0 else 50,
        eval_strategy="steps",
        eval_steps=100 if max_steps > 0 else 200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        load_best_model_at_end=(max_steps < 0),
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
    )

    cbs = [EscritaCallback()]
    if max_steps < 0:
        cbs.append(EarlyStoppingCallback(early_stopping_patience=3))

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        args=cfg,
        callbacks=cbs,
    )
    # Loss só na resposta (template nativo Qwen2.5)
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    ckpts = sorted(glob.glob(f"{out_dir}/checkpoint-*"))
    trainer.train(resume_from_checkpoint=bool(ckpts))

    elapsed_s = time.time() - t0
    peak_gb = torch.cuda.max_memory_allocated() / (1024**3)
    steps_done = max(1, trainer.state.global_step)
    print(f"Passos: {steps_done} | Tempo: {elapsed_s/60:.1f} min ({elapsed_s/steps_done:.2f}s/step) | VRAM pico: {peak_gb:.2f} GB")

    if max_steps < 0:
        model.save_pretrained(out_dir)
        tokenizer.save_pretrained(out_dir)
        hist = trainer.state.log_history
        with open(f"{out_dir}/loss_history.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=["step", "loss", "eval_loss"])
            w.writeheader()
            for h in hist:
                if "loss" in h or "eval_loss" in h:
                    w.writerow({"step": h.get("step"), "loss": h.get("loss", ""), "eval_loss": h.get("eval_loss", "")})
        tr = [(h["step"], h["loss"]) for h in hist if "loss" in h]
        ev = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
        plt.figure(figsize=(8, 4))
        if tr: plt.plot([x[0] for x in tr], [x[1] for x in tr], label="train_loss")
        if ev: plt.plot([x[0] for x in ev], [x[1] for x in ev], label="val_loss", marker="o")
        plt.title(f"Loss {os.path.basename(out_dir)}"); plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(True)
        plt.savefig(f"{out_dir}/loss_curve.png", dpi=120, bbox_inches="tight")
        plt.close()

    del trainer, model
    torch.cuda.empty_cache()
    return elapsed_s, steps_done, peak_gb


In [ ]:
# Célula 6: Teste curto (200 passos, lr=1e-4) para medir s/step, VRAM e estimativa total
elapsed_s, steps_done, peak_gb = run_training(1e-4, f"{DRIVE_ROOT}/out/dryrun-200", max_steps=200)
total_steps_1ep = len(train_data) // 16
est_min_per_lr = (elapsed_s / steps_done) * total_steps_1ep / 60.0
print(f"Estimativa para 1 época ({total_steps_1ep} steps) por LR na {gpu_name}: ~{est_min_per_lr:.1f} min")


In [ ]:
# Célula 7: Treino completo das 3 LRs sequencialmente (só rodar após aprovar o teste curto)
for lr in [5e-5, 1e-4, 2e-4]:
    lr_tag = f"lr-{lr:.0e}".replace("e-0", "e-")
    print(f"\n=== Treinando {lr_tag} ===")
    run_training(lr, f"{DRIVE_ROOT}/out/{lr_tag}", max_steps=-1)


In [ ]:
# Célula 8: Exportação GGUF Q8_0 + Modelfile (rodar SOMENTE para o vencedor após bench na validação)
WINNER_DIR = f"{DRIVE_ROOT}/out/lr-1e-4"  # ajustar para o vencedor
GGUF_OUT = f"{DRIVE_ROOT}/out/winner-gguf"

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=WINNER_DIR,
    max_seq_length=MAX_SEQ_LEN,
    dtype=DTYPE,
    load_in_4bit=False,
)
model.save_pretrained_gguf(GGUF_OUT, tokenizer, quantization_method="q8_0")

modelfile = f"""FROM ./unsloth.Q8_0.gguf
PARAMETER temperature 0
PARAMETER top_p 1
PARAMETER repeat_penalty 1.05
PARAMETER num_ctx 8192
SYSTEM "Você escreve funções Go. Responda só com o código completo da função pedida, sem explicação."
"""
with open(f"{GGUF_OUT}/Modelfile", "w") as f:
    f.write(modelfile)
print("GGUF Q8_0 e Modelfile salvos em:", GGUF_OUT)
